# DiabetesCare AI: Exploratory Data Analysis & Model Training

**Course / Project:** AI for Healthcare  
**Objective:** Build an explainable machine learning model to predict diabetes risk using clinical measurements from the Pima Indians Diabetes dataset, prioritizing **Recall** on positive cases and explaining decisions with **SHAP**.

---

## 1. Environment & Data Loading

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, roc_curve
)
import shap

# Load dataset
data_path = os.path.join("..", "data", "diabetes.csv")
df = pd.read_csv(data_path)
print(f"Dataset shape: {df.shape}")
df.head()

## 2. Exploratory Data Analysis & Physiological Zero Detection

In the Pima Indians dataset, values of 0 for physiological metrics like **Glucose, BloodPressure, SkinThickness, Insulin, and BMI** are physically impossible and indicate missing measurements.

In [ ]:
zero_cols = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
print("Percentage of invalid zeros per clinical feature:")
for col in zero_cols:
    zero_cnt = (df[col] == 0).sum()
    pct = (zero_cnt / len(df)) * 100
    print(f" - {col:15s}: {zero_cnt:3d} rows ({pct:.1f}%)")

## 3. Data Cleaning & Stratified Train-Test Split

We replace invalid zeros with `NaN` and perform a stratified 80/20 train/test split. Medians are calculated **strictly on the training split** to prevent data leakage.

In [ ]:
feature_cols = [
    "Pregnancies", "Glucose", "BloodPressure", "SkinThickness",
    "Insulin", "BMI", "DiabetesPedigreeFunction", "Age"
]
target_col = "Outcome"

X = df[feature_cols].copy()
y = df[target_col].copy()

# Replace zeros with NaN in physiological columns
for col in zero_cols:
    X[col] = X[col].replace(0, np.nan)

# Stratified 80/20 train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Fit medians strictly on X_train
train_medians = {col: float(X_train[col].median(skipna=True)) for col in zero_cols}
print("Training Medians for Imputation:", json.dumps(train_medians, indent=2))

# Impute training and test splits using train_medians
X_train_imp = X_train.copy()
X_test_imp = X_test.copy()
for col, med in train_medians.items():
    X_train_imp[col] = X_train_imp[col].fillna(med)
    X_test_imp[col] = X_test_imp[col].fillna(med)

print(f"X_train shape: {X_train_imp.shape}, X_test shape: {X_test_imp.shape}")

## 4. Hyperparameter Tuning via GridSearchCV

We optimize a `RandomForestClassifier` with 5-fold cross-validation. Since early detection is critical and false negatives are hazardous, we set `scoring='recall'` and `class_weight='balanced'`.

In [ ]:
rf_base = RandomForestClassifier(random_state=42, class_weight="balanced")

param_grid = {
    "n_estimators": [50, 100, 200],
    "max_depth": [None, 4, 6, 8],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2"]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid_search = GridSearchCV(
    estimator=rf_base,
    param_grid=param_grid,
    scoring="recall",
    cv=cv,
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train_imp, y_train)
best_model = grid_search.best_estimator_
print("Best Hyperparameters:", grid_search.best_params_)
print(f"Best 5-Fold CV Recall: {grid_search.best_score_:.4f}")

## 5. Model Evaluation on Held-Out Test Set

In [ ]:
y_pred = best_model.predict(X_test_imp)
y_prob = best_model.predict_proba(X_test_imp)[:, 1]

metrics = {
    "accuracy": round(float(accuracy_score(y_test, y_pred)), 4),
    "precision": round(float(precision_score(y_test, y_pred)), 4),
    "recall": round(float(recall_score(y_test, y_pred)), 4),
    "f1_score": round(float(f1_score(y_test, y_pred)), 4),
    "roc_auc": round(float(roc_auc_score(y_test, y_prob)), 4),
    "confusion_matrix": confusion_matrix(y_test, y_pred).tolist()
}

print("Test Set Performance:")
for k, v in metrics.items():
    print(f"  {k:18s}: {v}")

print("\nClassification Report:\n", classification_report(y_test, y_pred))

## 6. SHAP Explainability Analysis

We fit a SHAP `TreeExplainer` on the best Random Forest model to compute global feature importances and local patient-level attribution.

In [ ]:
explainer = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_test_imp)

# Handle multi-output vs single output format across SHAP versions
if isinstance(shap_values, list):
    shap_pos = shap_values[1]
elif len(shap_values.shape) == 3:
    shap_pos = shap_values[:, :, 1]
else:
    shap_pos = shap_values

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_pos, X_test_imp, feature_names=feature_cols, show=False)
plt.title("Global Feature Impact on Diabetes Risk (SHAP Beeswarm)", fontsize=14)
plt.tight_layout()
plt.show()